# 99 · 정리 (Cleanup) — 🔴 반드시 실행 — 텍스트 분류(intent)

**TL;DR** — endpoint와 모델, 그리고 배포했다면 AgentCore Runtime까지 삭제해 과금을 멈춥니다.

**Why** — real-time endpoint는 삭제하기 전까지 사용 여부와 무관하게 시간당 계속 과금되기 때문입니다.

**기존 Pain Point** — 실습 후 리소스를 정리하지 않으면 GPU 인스턴스 요금이 계속 청구됩니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib, boto3
from common import config, aws_utils; importlib.reload(config)
aws_utils.print_cost_warning()   # 🔴 config가 아니라 aws_utils에 있습니다
print('region:', config.AWS_REGION)

## 1. SageMaker endpoint + endpoint-config + model 삭제
endpoint를 배포하면 **endpoint · endpoint-config · model** 세 리소스가 함께 만들어집니다. 시간당 과금은 endpoint에서만 발생하지만, config와 model이 남으면 콘솔이 지저분해지고 계정당 개수 제한에도 걸립니다.

🔴 **model 이름은 `endpoint_name`과 다릅니다** — `ModelBuilder`가 `model-42c30d1e` 같은 임의 이름을 자동 생성하기 때문입니다. 그래서 `endpoint_name`으로 지우려 하면 model만 조용히 남습니다(실측). 아래 셀은 **endpoint-config에서 실제 ModelName을 조회**해 지웁니다.
🔴 삭제 순서는 **endpoint → config → model**입니다(config/model이 사용 중이면 삭제가 거부됨).

In [ ]:
# 트랙 전용 키 우선 — 전역 endpoint_name 은 다른 트랙이 덮어씁니다.
%store -r ep_classification
%store -r endpoint_name
endpoint_name = globals().get('ep_classification') or globals().get('endpoint_name')
assert endpoint_name, 'endpoint_name 이 없습니다 — 03의 배포 셀을 먼저 실행하세요.'
print('사용할 endpoint:', endpoint_name)
sm = boto3.client('sagemaker', region_name=config.AWS_REGION)

# 1) endpoint-config에서 실제 model 이름을 먼저 알아낸다(삭제하면 조회 불가 → 순서 중요).
model_names = []
try:
    cfg = sm.describe_endpoint_config(EndpointConfigName=endpoint_name)
    model_names = [v['ModelName'] for v in cfg.get('ProductionVariants', []) if v.get('ModelName')]
    print('endpoint-config가 참조하는 model:', model_names or '(없음)')
except Exception as e:
    print('endpoint-config 조회 생략:', str(e)[:120])

# 2) endpoint → endpoint-config → model 순서로 삭제(각각 독립적으로 감싸 하나가 실패해도 계속).
for fn, arg, name in ([(sm.delete_endpoint, 'EndpointName', endpoint_name),
                       (sm.delete_endpoint_config, 'EndpointConfigName', endpoint_name)]
                      + [(sm.delete_model, 'ModelName', m) for m in model_names]):
    try:
        fn(**{arg: name}); print(f'deleted: {arg}={name}')
    except Exception as e:
        print(f'skipped {arg}={name}: {str(e)[:110]}')

### 이 트랙의 남은 리소스 일괄 정리 (여러 번 배포했다면)
실습 중 endpoint를 여러 번 띄웠다면 `%store`의 `endpoint_name`은 **마지막 것만** 가리킵니다. 아래 셀로 이 트랙 prefix에 해당하는 잔여 리소스를 모두 찾아 정리하세요 — 이름 없는 model까지 포함합니다.

In [ ]:
# 🔴 이 트랙이 만든 리소스를 prefix로 훑어 남은 것을 모두 삭제합니다(다른 트랙/작업은 건드리지 않음).
PREFIX = 'gemma-classification'
DRY = True   # 먼저 True로 목록만 확인 → 맞으면 False로 바꿔 실제 삭제

eps = [e['EndpointName'] for e in sm.list_endpoints(NameContains=PREFIX)['Endpoints']]
cfgs = [c['EndpointConfigName'] for c in sm.list_endpoint_configs(NameContains=PREFIX)['EndpointConfigs']]
# model은 ModelBuilder가 임의 이름(model-xxxx)을 붙여 prefix로 못 찾는다 → config가 참조하는 것을 모은다.
models = set()
for c in cfgs:
    try:
        for v in sm.describe_endpoint_config(EndpointConfigName=c).get('ProductionVariants', []):
            if v.get('ModelName'):
                models.add(v['ModelName'])
    except Exception:
        pass
print('endpoints       :', eps or '(없음)')
print('endpoint-configs:', cfgs or '(없음)')
print('models          :', sorted(models) or '(없음)')

if DRY:
    print('\nDRY=True — 목록만 표시했습니다. 위 목록이 맞으면 DRY=False로 바꿔 다시 실행하세요.')
else:
    for n in eps:
        try: sm.delete_endpoint(EndpointName=n); print('deleted endpoint:', n)
        except Exception as e: print('skip endpoint', n, str(e)[:80])
    for n in cfgs:
        try: sm.delete_endpoint_config(EndpointConfigName=n); print('deleted config:', n)
        except Exception as e: print('skip config', n, str(e)[:80])
    for n in sorted(models):
        try: sm.delete_model(ModelName=n); print('deleted model:', n)
        except Exception as e: print('skip model', n, str(e)[:80])

## 2. (02b를 실행했다면) 로컬 리소스 정리 — 모델 파일·vLLM 프로세스
`02b_local_serve`로 로컬 검증을 했다면 **내 머신에** 다음이 남아 있습니다. 과금은 없지만 디스크와 GPU를 계속 차지합니다:

| 남는 것 | 크기 | 왜 지워야 하나 |
|---|---|---|
| `local_model/` | **약 15GB**(E4B) | 모델 압축 해제본 |
| vLLM 서버 프로세스 | GPU 전체 | 살아 있으면 **다음 학습/서빙이 OOM** |
| `bench/`, `req.json` | 작음 | 벤치 결과·curl payload |

터미널에서 한 줄로 정리합니다(**목록만 보여주는 게 기본** — `--yes`를 줘야 실제 삭제):
```bash
bash scripts/cleanup_local.sh              # 무엇이 지워질지 먼저 확인
bash scripts/cleanup_local.sh --yes        # 실제 삭제
KEEP_MODEL=1 bash scripts/cleanup_local.sh --yes   # 모델은 남기고 나머지만(재검증 예정)
```
🔴 vLLM 종료는 `kill <pid>`로 정밀하게 합니다 — `pkill -f vllm`은 실행 중인 셸/노트북까지 죽일 수 있습니다.

In [ ]:
# (참고) 지금 로컬에 무엇이 남아 있는지 노트북에서 확인 — 삭제는 위 터미널 명령으로.
import os, shutil, subprocess
for p in ('local_model', 'bench', 'req.json', 'model.tar.gz'):
    if os.path.exists(p):
        sz = subprocess.run(['du', '-sh', p], capture_output=True, text=True).stdout.split()[0]
        print(f'  {sz:>8}  {p}')
    else:
        print(f'  {"-":>8}  {p} (없음)')
print()
# GPU를 아직 물고 있는 프로세스가 있는지
try:
    out = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,used_memory',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    print('GPU 점유 프로세스:', out or '(없음 ✅)')
except FileNotFoundError:
    print('(nvidia-smi 없음 — GPU 없는 환경)')
print('\n정리:  bash scripts/cleanup_local.sh --yes')

## 3. (AgentCore 사용 시) 로컬 dev·프로젝트 + AWS Runtime 정리
06번에서 AgentCore를 썼다면, endpoint와는 별도로 로컬 dev 서버·프로젝트 폴더와(배포했다면) Runtime·ECR도 정리해야 합니다. **터미널에서** 아래 cleanup 스크립트 한 줄로 처리합니다:
```bash
bash agentcore/cleanup_agent.sh          # 로컬 정리(dev 프로세스 종료 + 프로젝트 폴더 삭제)
bash agentcore/cleanup_agent.sh --aws    # 배포까지 했다면: 로컬 + AWS Runtime/ECR(agentcore destroy)
```
> 🔴 로컬 dev만 돌렸으면(배포 안 함) 첫 줄로 충분합니다 — AWS엔 과금 리소스가 없습니다. `agentcore deploy`로 클라우드에 올렸을 때만 `--aws`가 필요합니다.

In [ ]:
# (참고) 노트북에서 로컬 정리를 실행하려면 — 리포 루트 기준 절대경로로:
import os, subprocess
def _find_repo(d=None):
    d = os.path.abspath(d or os.getcwd())
    for _ in range(6):
        if os.path.isfile(os.path.join(d, 'agentcore', 'cleanup_agent.sh')):
            return d
        d = os.path.dirname(d)
    return None
_repo = _find_repo()
if _repo:
    print('cleanup:', os.path.join(_repo, 'agentcore', 'cleanup_agent.sh'))
    print('터미널 실행 권장:  bash agentcore/cleanup_agent.sh [--aws]')
    # subprocess.run(['bash', os.path.join(_repo,'agentcore','cleanup_agent.sh')], check=False)
else:
    print('agentcore/cleanup_agent.sh를 못 찾음 — 리포 루트에서 터미널로 실행하세요')

## 4. 확인 — 과금 리소스가 모두 사라졌는지
endpoint뿐 아니라 endpoint-config·model까지 함께 확인합니다. **시간당 과금은 endpoint에서만** 발생하므로 그 목록이 비어 있으면 요금은 멈춘 것입니다(config/model은 개수 제한과 콘솔 정리 문제).
🔴 **이 트랙 것과 다른 트랙 것을 구분해서** 보여 줍니다 — 계정 전체 목록만 보면 다른 트랙의 endpoint를 보고 '이 트랙이 안 지워졌다'고 오해하게 됩니다.

In [ ]:
eps = [e['EndpointName'] for e in sm.list_endpoints()['Endpoints']]
cfgs = [c['EndpointConfigName'] for c in sm.list_endpoint_configs()['EndpointConfigs']]
mdls = [m['ModelName'] for m in sm.list_models()['Models']]
mine = [n for n in eps if n.startswith('gemma-classification')]
others = [n for n in eps if n not in mine]

print('이 트랙(gemma-classification) endpoint :', mine or 'none ✅  ← 이 트랙 과금 멈춤')
print('다른 트랙/작업 endpoint      :', others or 'none')
print('endpoint-configs(계정 전체) :', cfgs or 'none ✅')
print('models(계정 전체)           :', mdls or 'none ✅')
print()
print('region:', config.AWS_REGION, ' ← 다른 리전에도 띄운 적이 있으면 그 리전도 확인하세요')
if mine:
    print('\n🔴 이 트랙 endpoint가 남아 있습니다 — 위 §1의 일괄 정리 셀을 DRY=False로 실행하세요.')
elif others:
    print('\n⚠️ 이 트랙은 정리됐습니다. 다른 트랙 endpoint가 과금 중이니 그 트랙의 99_cleanup도 실행하세요:')
    for n in others:
        print('   -', n)
else:
    print('\n✅ 이 리전에 남은 endpoint가 없습니다 — 과금 멈춤.')

✅ 정리가 끝났습니다. Bedrock Converse는 상시 유지되는 리소스가 없고 호출 단위로만 과금되므로 별도의 teardown이 필요하지 않습니다.